# Day 044 — Exercise 4: update_price

**What you'll build:** `update_price(session, item_id, new_price) -> Item | None` — fetch an item by primary key, update its price, commit, and return the updated item. Return `None` if the id does not exist.

**Why it matters:** `session.get(Model, pk)` is the efficient ORM method for lookup by primary key — it checks the session's identity map first (no SQL if already loaded), then queries the DB. Mutating the attribute directly (`item.price = new_price`) marks the object as 'dirty'; the next `commit()` generates the UPDATE statement automatically.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sqlalchemy import create_engine, String, Float, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Item(Base):
    __tablename__ = 'items'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    name:     Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    price:    Mapped[float] = mapped_column()
    quantity: Mapped[int]   = mapped_column(default=0)

    def __repr__(self):
        return f'Item(id={self.id}, name={self.name!r}, price={self.price})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def add_item(session, name, category, price, quantity=0):
    item = Item(name=name, category=category, price=price, quantity=quantity)
    session.add(item)
    session.commit()
    session.refresh(item)
    return item


def get_items(session, category=None):
    stmt = select(Item)
    if category is not None:
        stmt = stmt.where(Item.category == category)
    return list(session.execute(stmt).scalars().all())


engine  = setup_engine()
session = Session(engine)

laptop  = add_item(session, 'Laptop', 'Electronics', 999.99, 5)
chair   = add_item(session, 'Desk Chair', 'Furniture', 349.00, 3)

## Your Implementation

In [ ]:
def update_price(session, item_id, new_price):
    """
    Update the price of an item by id. Return None if not found.

    Steps:
    1. item = session.get(Item, item_id)  — None if not found
    2. if item is None: return None
    3. item.price = new_price
    4. session.commit()
    5. session.refresh(item)
    6. return item
    """
    # TODO: item = session.get(Item, item_id)
    # TODO: if item is None: return None
    # TODO: item.price = new_price
    # TODO: session.commit()
    # TODO: session.refresh(item)
    # TODO: return item
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'update_price' in globals()
        passed += 1; print('\u2705 Check 1: update_price is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns the updated Item
    try:
        updated = update_price(session, laptop.id, 899.99)
        assert isinstance(updated, Item), \
            f'expected Item, got {type(updated).__name__}'
        passed += 1; print('\u2705 Check 2: returns an Item')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: price was updated
    try:
        assert abs(updated.price - 899.99) < 0.01, \
            f'expected 899.99, got {updated.price}'
        passed += 1; print(f'\u2705 Check 3: price updated to {updated.price}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: change is persisted (re-fetch from DB)
    try:
        re_fetched = session.get(Item, laptop.id)
        assert abs(re_fetched.price - 899.99) < 0.01, \
            f'persisted price mismatch: {re_fetched.price}'
        passed += 1; print('\u2705 Check 4: change persisted in DB')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: non-existent id returns None
    try:
        result = update_price(session, 99999, 1.0)
        assert result is None, f'expected None for missing id, got {result}'
        passed += 1; print('\u2705 Check 5: missing id returns None')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def update_price(session, item_id, new_price):
    item = session.get(Item, item_id)
    if item is None:
        return None
    item.price = new_price
    session.commit()
    session.refresh(item)
    return item
```

</details>